# Drift Detector Evaluation

This notebook evaluates three drift detectors on synthetic time series data.
The detectors are ADWIN, KSWIN, and Page-Hinkley.
The results are used to create Table 1.

## Import Libraries

The required libraries are imported here.


In [1]:
import numpy as np
import pandas as pd
from river import drift

## ADWIN Detector

This function runs the ADWIN detector.
It calculates detection delay, false alarms, and missed detections.


In [2]:
def adwin_detector(data, true_points):

    adwin = drift.ADWIN(delta=0.002)
    adwin_points = []

    for i in range(len(data)):
        x = data["y"].iloc[i]
        adwin.update(x)
        if adwin.drift_detected:
            adwin_points.append(data["t"].iloc[i])

    matched_points = []
    delays = []

    for true_p in true_points:
        matched = None
        for d in adwin_points:
            if d >= true_p and d <= true_p + 500:
                matched = d
                break
        if matched is not None:
            matched_points.append(matched)
            delays.append(matched - true_p)

    false_points = []

    for d in adwin_points:
        if d not in matched_points:
            false_points.append(d)
    false_rate = len(false_points) / len(data) * 10000
    missed = len(true_points) - len(matched_points)
    if len(delays) > 0:
        average_delay = np.mean(delays)
    else:
        average_delay = np.nan

    return average_delay, false_rate, missed

## KSWIN Detector

This function runs the KSWIN detector.
It calculates detection delay, false alarms, and missed detections.


In [3]:
def kswin_detector(data, true_points):

    kswin = drift.KSWIN(
        alpha=0.005,
        window_size=200,
        stat_size=50,
        seed=42
    )

    kswin_points = []

    for i in range(len(data)):
        x = data["y"].iloc[i]
        kswin.update(x)
        if kswin.drift_detected:
            kswin_points.append(data["t"].iloc[i])

    matched_points = []
    delays = []

    for true_p in true_points:
        matched = None
        for d in kswin_points:
            if d >= true_p and d <= true_p + 500:
                matched = d
                break
        if matched is not None:
            matched_points.append(matched)
            delays.append(matched - true_p)

    false_points = []

    for d in kswin_points:
        if d not in matched_points:
            false_points.append(d)

    false_rate = len(false_points) / len(data) * 10000
    missed = len(true_points) - len(matched_points)

    if len(delays) > 0:
        average_delay = np.mean(delays)
    else:
        average_delay = np.nan

    return average_delay, false_rate, missed

## Page-Hinkley Detector

This function runs the Page-Hinkley detector.
It calculates detection delay, false alarms, and missed detections.


In [4]:
def page_hinkley_detector(data, true_points):

    ph = drift.PageHinkley(
        min_instances=30,
        delta=0.005,
        threshold=50
    )

    ph_points = []

    for i in range(len(data)):
        x = data["y"].iloc[i]
        ph.update(x)
        if ph.drift_detected:
            ph_points.append(data["t"].iloc[i])

    matched_points = []
    delays = []

    for true_p in true_points:
        matched = None
        for d in ph_points:
            if d >= true_p and d <= true_p + 500:
                matched = d
                break
        if matched is not None:
            matched_points.append(matched)
            delays.append(matched - true_p)

    false_points = []

    for d in ph_points:
        if d not in matched_points:
            false_points.append(d)

    false_rate = len(false_points) / len(data) * 10000
    missed = len(true_points) - len(matched_points)

    if len(delays) > 0:
        average_delay = np.mean(delays)
    else:
        average_delay = np.nan

    return average_delay, false_rate, missed

## Run Detectors on All Data Files

This function runs all three detectors on the synthetic data files.
It uses sudden, gradual, recurring, and no drift data.
Five seeds are used for each drift type.


In [5]:
def run_all_files():

    drift_types = ["sudden", "gradual", "recurring","none"]
    seeds = [1, 2, 3, 4, 5]
    results = []

    for drift_type in drift_types:
        for seed in seeds:

            file_name = "data/series_" + drift_type + "_n20000_seed" + str(seed) + ".csv"
            data = pd.read_csv(file_name)
            true_points = data.loc[
                data["is_changepoint"] == 1,
                "t"
            ].tolist()

            adwin_delay, adwin_false, adwin_missed = adwin_detector(
                data,
                true_points
            )

            results.append({
                "Detector": "ADWIN",
                "Drift type": drift_type,
                "Seed": seed,
                "Delay": adwin_delay,
                "False alarms / 10k": adwin_false,
                "Missed": adwin_missed,
                "Threshold": "delta = 0.002"
            })

            kswin_delay, kswin_false, kswin_missed = kswin_detector(
                data,
                true_points
            )

            results.append({
                "Detector": "KSWIN",
                "Drift type": drift_type,
                "Seed": seed,
                "Delay": kswin_delay,
                "False alarms / 10k": kswin_false,
                "Missed": kswin_missed,
                "Threshold": "alpha = 0.005"
            })

            ph_delay, ph_false, ph_missed = page_hinkley_detector(
                data,
                true_points
            )

            results.append({
                "Detector": "Page-Hinkley",
                "Drift type": drift_type,
                "Seed": seed,
                "Delay": ph_delay,
                "False alarms / 10k": ph_false,
                "Missed": ph_missed,
                "Threshold": "threshold = 50"
            })


    results = pd.DataFrame(results)

    return results

## View Detector Results

The detectors are run on all data files.
The results for each seed are displayed below.


In [6]:
results = run_all_files()

results

,Detector,Drift type,Seed,Delay,False alarms / 10k,Missed,Threshold
0,ADWIN,sudden,1,15.0,0.5,0,delta = 0.002
1,KSWIN,sudden,1,18.0,0.0,0,alpha = 0.005
2,Page-Hinkley,sudden,1,11.0,312.0,0,threshold = 50
3,ADWIN,sudden,2,15.0,3.5,0,delta = 0.002
4,KSWIN,sudden,2,18.0,0.0,0,alpha = 0.005
5,Page-Hinkley,sudden,2,11.0,312.0,0,threshold = 50
6,ADWIN,sudden,3,15.0,4.0,0,delta = 0.002
7,KSWIN,sudden,3,16.0,0.5,0,alpha = 0.005
8,Page-Hinkley,sudden,3,12.0,312.0,0,threshold = 50
9,ADWIN,sudden,4,15.0,2.0,0,delta = 0.002


## Create Table 1

The results are grouped by detector and drift type.
The mean delay and standard deviation are calculated.
The average false alarms and total missed detections are also calculated.
The final results are shown as Table 1.


In [7]:
table1_rows = []

groups = results.groupby(["Detector", "Drift type"])

for (detector, drift_type), group in groups:

    mean_delay = group["Delay"].mean()
    sd_delay = group["Delay"].std()
    mean_false = group["False alarms / 10k"].mean()
    total_missed = group["Missed"].sum()
    threshold = group["Threshold"].iloc[0]
    number_seeds = len(group)
    delay_result = (
        str(round(mean_delay, 2))
        + " +/- "
        + str(round(sd_delay, 2))
    )

    table1_rows.append({
        "Detector": detector,
        "Drift type": drift_type,
        "Delay (mean +/- SD)": delay_result,
        "False alarms / 10k": round(mean_false, 2),
        "Missed": total_missed,
        "Threshold": threshold,
        "Seeds": number_seeds
    })


table1 = pd.DataFrame(table1_rows)

table1

,Detector,Drift type,Delay (mean +/- SD),False alarms / 10k,Missed,Threshold,Seeds
0,ADWIN,gradual,207.0 +/- 0.0,3.7,0,delta = 0.002,5
1,ADWIN,none,nan +/- nan,2.5,0,delta = 0.002,5
2,ADWIN,recurring,15.5 +/- 0.0,3.8,0,delta = 0.002,5
3,ADWIN,sudden,15.0 +/- 0.0,2.8,0,delta = 0.002,5
4,KSWIN,gradual,122.0 +/- nan,0.1,4,alpha = 0.005,5
5,KSWIN,none,nan +/- nan,0.0,0,alpha = 0.005,5
6,KSWIN,recurring,15.7 +/- 1.96,0.0,0,alpha = 0.005,5
7,KSWIN,sudden,17.6 +/- 0.89,0.1,0,alpha = 0.005,5
8,Page-Hinkley,gradual,17.2 +/- 0.45,311.5,0,threshold = 50,5
9,Page-Hinkley,none,nan +/- nan,312.0,0,threshold = 50,5
